In [4]:
import pandas as pd

dataset = pd.read_csv("../curated_dataset.csv")

In [24]:
import pandas as pd 

df = pd.read_csv("pdb_chemical_components_curated_dataset.csv")
print(df.chem_comp.unique().shape)
df.dropna(subset=['chem_comp'], inplace=True)
print(df.chem_comp.unique().shape)

(2235,)
(2234,)


In [25]:
# for compound_id in compound_ids:
import requests


def query_for_smiles(component_ids):
    #base graphql query:
    base_url = 'https://data.rcsb.org/graphql'
    
    #all graphql queries require format: ["id1", "id2", so on...]
    query_fmt = '[' + ', '.join(['"' + i + '"' for i in component_ids]) + ']'
    
    #put the query together
    graphql_query = """
    {
      chem_comps(comp_ids:""" + query_fmt +""") {
        chem_comp {
          id
        }
        rcsb_chem_comp_descriptor {
          SMILES_stereo
        }
      }
    }
    """
    
    #encode as an html request and get the json data
    r = requests.post(base_url, json={'query': graphql_query})
    output = r.json()
    return output

def get_smiles(chem_comp):
    chem_comp = list(set(chem_comp))
    output = query_for_smiles(chem_comp)
    names = list()
    smiles = list()
    for i in output['data']['chem_comps']:
        name = i['chem_comp']['id']
        if i['rcsb_chem_comp_descriptor'] is None:
            smi = None
        else:
            smi = i['rcsb_chem_comp_descriptor']['SMILES_stereo']
        names.append(name)
        smiles.append(smi)
    return pd.DataFrame({'chem_comp':names, 'smiles':smiles})

df = df.merge(get_smiles(df['chem_comp']), on='chem_comp')

In [5]:
# get the inchikeys
from rdkit import Chem
from rdkit.Chem import AllChem

def smiles_to_inchikey(smiles):
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    inchikey = AllChem.MolToInchiKey(mol)
    return inchikey

df['inchikey'] = df['smiles'].apply(smiles_to_inchikey)
# separate in different cells for clarity by -
df["inchikey_skeleton"] = df['inchikey'].apply(lambda x: x.split('-')[0] if pd.notna(x) else None)
df["inchikey_stereochemistry"] = df['inchikey'].apply(lambda x: '-'.join(x.split('-')[:2]) if pd.notna(x) else None)



NameError: name 'df' is not defined

In [47]:
df.dropna(subset=["smiles", "inchikey"], inplace=True)

In [48]:
df.to_csv("pdb_chemical_components_curated_dataset_with_smiles.csv", index=False)

In [31]:
import pandas as pd

dataset = pd.read_csv("curated_dataset.csv")

unique_compounds = dataset[["Substrate ID", "SMILES"]].drop_duplicates().reset_index(drop=True)
unique_compounds["inchikey"] = unique_compounds['SMILES'].apply(smiles_to_inchikey)
unique_compounds["inchikey_skeleton"] = unique_compounds['inchikey'].apply(lambda x: x.split('-')[0] if pd.notna(x) else None)
unique_compounds["inchikey_stereochemistry"] = unique_compounds['inchikey'].apply(lambda x: '-'.join(x.split('-')[:2]) if pd.notna(x) else None)
unique_compounds

[15:21:46] WARNING: not removing hydrogen atom without neighbors


,Substrate ID,SMILES,inchikey,inchikey_skeleton,inchikey_stereochemistry
0,compound_aminotransferase_dataset 1,C(C(=O)O)N,DHMQDGOQFOQNFH-UHFFFAOYSA-N,DHMQDGOQFOQNFH,DHMQDGOQFOQNFH-UHFFFAOYSA
1,compound_aminotransferase_dataset 2,C[C@@H](C(=O)O)N,QNAYBMKLOCPYGJ-REOHCLBHSA-N,QNAYBMKLOCPYGJ,QNAYBMKLOCPYGJ-REOHCLBHSA
2,compound_aminotransferase_dataset 3,CC(C)[C@@H](C(=O)O)N,KZSNJWFQEVHDMF-BYPYZUCNSA-N,KZSNJWFQEVHDMF,KZSNJWFQEVHDMF-BYPYZUCNSA
3,compound_aminotransferase_dataset 4,CC(C)C[C@@H](C(=O)O)N,ROHFNLRQFUQHCH-YFKPBYRVSA-N,ROHFNLRQFUQHCH,ROHFNLRQFUQHCH-YFKPBYRVSA
4,compound_aminotransferase_dataset 5,CC[C@H](C)[C@@H](C(=O)O)N,AGPKZVBTJJNPAG-WHFBIAKZSA-N,AGPKZVBTJJNPAG,AGPKZVBTJJNPAG-WHFBIAKZSA
...,...,...,...,...,...
1723,heptynoate,O=[N+](C1=CC=C(OC(CCCCC#C)=O)C=C1)[O-],NEMGDMRNWPMDBA-UHFFFAOYSA-N,NEMGDMRNWPMDBA,NEMGDMRNWPMDBA-UHFFFAOYSA
1724,hexanoate,CCCCCC(OC1=CC=C([N+]([O-])=O)C=C1)=O,OLRXUEYZKCCEKK-UHFFFAOYSA-N,OLRXUEYZKCCEKK,OLRXUEYZKCCEKK-UHFFFAOYSA
1725,oxidazole,O=C(OC1=CC=C([N+]([O-])=O)C=C1)CCC2=NN=C(C3=CC...,BDEWALBWCQVRGC-UHFFFAOYSA-N,BDEWALBWCQVRGC,BDEWALBWCQVRGC-UHFFFAOYSA
1726,TMA,O=[N+](C1=CC=C(OC(C(C)(C)C)=O)C=C1)[O-],QADVJDGFQGNSIF-UHFFFAOYSA-N,QADVJDGFQGNSIF,QADVJDGFQGNSIF-UHFFFAOYSA


In [32]:
# see intersection between both datasets
curated_inchikeys = set(unique_compounds['inchikey'].dropna().unique())
pdb_inchikeys = set(df['inchikey'].dropna().unique())
intersection = curated_inchikeys.intersection(pdb_inchikeys)
print(f"Number of unique compounds in curated dataset: {len(curated_inchikeys)}")
print(f"Number of unique compounds in pdb dataset: {len(pdb_inchikeys)}")
print(f"Number of compounds in intersection: {len(intersection)}")

Number of unique compounds in curated dataset: 1691
Number of unique compounds in pdb dataset: 2211
Number of compounds in intersection: 260


In [60]:
# see intersection between both datasets
curated_inchikeys = set(unique_compounds['inchikey_skeleton'].dropna().unique())
pdb_inchikeys = set(df['inchikey_skeleton'].dropna().unique())
intersection = curated_inchikeys.intersection(pdb_inchikeys)
print(f"Number of unique compounds in curated dataset: {len(curated_inchikeys)}")
print(f"Number of unique compounds in pdb dataset: {len(pdb_inchikeys)}")
print(f"Number of compounds in intersection: {len(intersection)}")

Number of unique compounds in curated dataset: 1354
Number of unique compounds in pdb dataset: 2107
Number of compounds in intersection: 438


In [61]:
# see intersection between both datasets
curated_inchikeys = set(unique_compounds['inchikey_stereochemistry'].dropna().unique())
pdb_inchikeys = set(df['inchikey_stereochemistry'].dropna().unique())
intersection = curated_inchikeys.intersection(pdb_inchikeys)
print(f"Number of unique compounds in curated dataset: {len(curated_inchikeys)}")
print(f"Number of unique compounds in pdb dataset: {len(pdb_inchikeys)}")
print(f"Number of compounds in intersection: {len(intersection)}")

Number of unique compounds in curated dataset: 1556
Number of unique compounds in pdb dataset: 2199
Number of compounds in intersection: 413


In [62]:
df[df["inchikey_stereochemistry"].isin(intersection)]

,Uniprot_ID,PDB_ID,nonpol_ID,chem_comp,smiles,inchikey,inchikey_skeleton,inchikey_stereochemistry
0,P00509,1AAM,1AAM_2,SO4,[O-]S(=O)(=O)[O-],QAOWNCQODCNURD-UHFFFAOYSA-L,QAOWNCQODCNURD,QAOWNCQODCNURD-UHFFFAOYSA
1,P00509,1AHE,1AHE_2,SO4,[O-]S(=O)(=O)[O-],QAOWNCQODCNURD-UHFFFAOYSA-L,QAOWNCQODCNURD,QAOWNCQODCNURD-UHFFFAOYSA
2,P00509,1AHF,1AHF_4,SO4,[O-]S(=O)(=O)[O-],QAOWNCQODCNURD-UHFFFAOYSA-L,QAOWNCQODCNURD,QAOWNCQODCNURD-UHFFFAOYSA
3,Q9KPM2,3N28,3N28_2,SO4,[O-]S(=O)(=O)[O-],QAOWNCQODCNURD-UHFFFAOYSA-L,QAOWNCQODCNURD,QAOWNCQODCNURD-UHFFFAOYSA
4,Q9X0Y1,3KBB,3KBB_2,SO4,[O-]S(=O)(=O)[O-],QAOWNCQODCNURD-UHFFFAOYSA-L,QAOWNCQODCNURD,QAOWNCQODCNURD-UHFFFAOYSA
...,...,...,...,...,...,...,...,...
17674,Q6BF16,2V82,2V82_2,KDP,C([C@H]([C@@H](COP(=O)(O)O)O)O)C(=O)C(=O)O,OVPRPPOVAXRCED-NQXXGFSBSA-N,OVPRPPOVAXRCED,OVPRPPOVAXRCED-NQXXGFSBSA
17681,P32719,3CTL,3CTL_2,S6P,C([C@@H]([C@H]([C@@H]([C@@H](COP(=O)(O)O)O)O)O...,GACTWZZMVMUKNG-SLPGGIOYSA-N,GACTWZZMVMUKNG,GACTWZZMVMUKNG-SLPGGIOYSA
17712,P32169,1OJR,1OJR_4,2HA,C(C(=O)CO)O,RXKJFZQQPQGTFL-UHFFFAOYSA-N,RXKJFZQQPQGTFL,RXKJFZQQPQGTFL-UHFFFAOYSA
17714,Q7WG29,3L8H,3L8H_7,SAR,CNCC(=O)O,FSYKKLYZXJSNPZ-UHFFFAOYSA-N,FSYKKLYZXJSNPZ,FSYKKLYZXJSNPZ-UHFFFAOYSA


In [63]:
dataset = dataset.merge(unique_compounds, left_on='SMILES', right_on='SMILES', how='left')
dataset

,Unnamed: 0,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,Validated,RHEA_ID,EC number,reaction_SMILES,inchikey,inchikey_skeleton,inchikey_stereochemistry
0,0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C(C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,DHMQDGOQFOQNFH-UHFFFAOYSA-N,DHMQDGOQFOQNFH,DHMQDGOQFOQNFH-UHFFFAOYSA
1,1,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C[C@@H](C(=O)O)N,1.0,P00509,compound_aminotransferase_dataset 2,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,QNAYBMKLOCPYGJ-REOHCLBHSA-N,QNAYBMKLOCPYGJ,QNAYBMKLOCPYGJ-REOHCLBHSA
2,2,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC(C)[C@@H](C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 3,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,KZSNJWFQEVHDMF-BYPYZUCNSA-N,KZSNJWFQEVHDMF,KZSNJWFQEVHDMF-BYPYZUCNSA
3,3,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC(C)C[C@@H](C(=O)O)N,1.0,P00509,compound_aminotransferase_dataset 4,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,ROHFNLRQFUQHCH-YFKPBYRVSA-N,ROHFNLRQFUQHCH,ROHFNLRQFUQHCH-YFKPBYRVSA
4,4,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC[C@H](C)[C@@H](C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 5,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,AGPKZVBTJJNPAG-WHFBIAKZSA-N,AGPKZVBTJJNPAG,AGPKZVBTJJNPAG-WHFBIAKZSA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73225,76466,MKYRAVTLESFGYQLAPVVVSTSDLEARLEPLYRQLRIAPGQLQAM...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Pelobacter_propionicus,benzoate,https://doi.org/10.1093/synbio/ysaa004,True,NaN,NaN,NaN,GMKZBFFLCONHDE-UHFFFAOYSA-N,GMKZBFFLCONHDE,GMKZBFFLCONHDE-UHFFFAOYSA
73226,76467,MAFLSVNNVEIVGLAAAVPKNVETLDNLEFFAPGEAEKVMALTGIK...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Bacteroides_luti,benzoate,https://doi.org/10.1093/synbio/ysaa004,True,NaN,NaN,NaN,GMKZBFFLCONHDE-UHFFFAOYSA-N,GMKZBFFLCONHDE,GMKZBFFLCONHDE-UHFFFAOYSA
73227,76468,MSAPRYSQVSAVAVRLPDEDLTTPELEELLAERNPRVDVPRGLIER...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Nonomuraea_candida,benzoate,https://doi.org/10.1093/synbio/ysaa004,True,NaN,NaN,NaN,GMKZBFFLCONHDE-UHFFFAOYSA-N,GMKZBFFLCONHDE,GMKZBFFLCONHDE-UHFFFAOYSA
73228,76469,MRYQRVFINKIAYELPKEKVATSFLEEQLTDVYQELGIPLGQVEAL...,C1=CC=C(C=C1)C(=O)OC2=CC=C(C=C2)[N+](=O)[O-],0.0,Legionella_brunensis,benzoate,https://doi.org/10.1093/synbio/ysaa004,True,NaN,NaN,NaN,GMKZBFFLCONHDE-UHFFFAOYSA-N,GMKZBFFLCONHDE,GMKZBFFLCONHDE-UHFFFAOYSA


In [64]:
dataset.columns = dataset.columns.str.replace('Enzyme ID', 'Uniprot_ID')

In [ ]:
intersection = pd.merge(dataset, df, on=['Uniprot_ID', 'inchikey_stereochemistry'], how='inner')
intersection

,Unnamed: 0,Sequence,SMILES,Binding,Uniprot_ID,Substrate ID,Publication,Validated,RHEA_ID,EC number,reaction_SMILES,inchikey_x,inchikey_skeleton_x,inchikey_stereochemistry,PDB_ID,nonpol_ID,chem_comp,smiles,inchikey_y,inchikey_skeleton_y
0,6,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C1=CC(=CC=C1C[C@@H](C(=O)O)N)O,1.0,P00509,compound_aminotransferase_dataset 7,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,OUYCCCASQSFEME-QMMMGPOBSA-N,OUYCCCASQSFEME,OUYCCCASQSFEME-QMMMGPOBSA,1AHG,1AHG_2,TYR,c1cc(ccc1C[C@@H](C(=O)O)N)O,OUYCCCASQSFEME-QMMMGPOBSA-N,OUYCCCASQSFEME
1,65914,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,N[C@@H](Cc1ccc(O)cc1)C(=O)O,1.0,P00509,CHEBI:58315,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,OUYCCCASQSFEME-QMMMGPOBSA-N,OUYCCCASQSFEME,OUYCCCASQSFEME-QMMMGPOBSA,1AHG,1AHG_2,TYR,c1cc(ccc1C[C@@H](C(=O)O)N)O,OUYCCCASQSFEME-QMMMGPOBSA-N,OUYCCCASQSFEME
2,18095,MLYIFDLGNVIVDIDFNRVLGAWSDLTRIPLASLKKSFHMGEAFHQ...,O=P(O)(O)O[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@...,1.0,P0A8Y3,compound_phospatase_dataset 156,https://doi.org/10.1073/pnas.1423570112,True,NaN,NaN,NaN,HXXFSFRBOHSIMQ-VFUOTHLCSA-N,HXXFSFRBOHSIMQ,HXXFSFRBOHSIMQ-VFUOTHLCSA,2B0C,2B0C_2,G1P,C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)OP(=O)(O)...,HXXFSFRBOHSIMQ-VFUOTHLCSA-N,HXXFSFRBOHSIMQ
3,70880,MLYIFDLGNVIVDIDFNRVLGAWSDLTRIPLASLKKSFHMGEAFHQ...,O=P([O-])([O-])O[C@H]1O[C@H](CO)[C@@H](O)[C@H]...,1.0,P0A8Y3,CHEBI:58601,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,HXXFSFRBOHSIMQ-VFUOTHLCSA-L,HXXFSFRBOHSIMQ,HXXFSFRBOHSIMQ-VFUOTHLCSA,2B0C,2B0C_2,G1P,C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)OP(=O)(O)...,HXXFSFRBOHSIMQ-VFUOTHLCSA-N,HXXFSFRBOHSIMQ
4,28553,MSTPRQILAAIFDMDGLLIDSEPLWDRAELDVMASLGVDISRRNEL...,O=C(O)COP(=O)(O)O,0.0,P77247,compound_phospatase_dataset 54,https://doi.org/10.1073/pnas.1423570112,True,NaN,NaN,NaN,ASCFNMCAHFUBCO-UHFFFAOYSA-N,ASCFNMCAHFUBCO,ASCFNMCAHFUBCO-UHFFFAOYSA,1TE2,1TE2_2,PGA,C(C(=O)O)OP(=O)(O)O,ASCFNMCAHFUBCO-UHFFFAOYSA-N,ASCFNMCAHFUBCO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408,71772,MAAAPRAGRRRGQPLLALLLLLLAPLPPGAPPGADAYFPEERWSPE...,O=c1nc([O-])ccn1[C@@H]1O[C@H](COP(=O)([O-])OP(...,1.0,Q8NBJ5,CHEBI:66914,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,HSCJRCZFDFQWRP-ABVWGUQPSA-L,HSCJRCZFDFQWRP,HSCJRCZFDFQWRP-ABVWGUQPSA,8ZGC,8ZGC_9,GDU,C1=CN(C(=O)NC1=O)[C@H]2[C@@H]([C@@H]([C@H](O2)...,HSCJRCZFDFQWRP-ABVWGUQPSA-N,HSCJRCZFDFQWRP
1409,71800,MQWQTKLPLIAILRGITPDEALAHVGAVIDAGFDAVEIPLNSPQWE...,O=C([O-])C(=O)C[C@@H](O)[C@H](O)COP(=O)([O-])[O-],1.0,Q6BF16,CHEBI:58298,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,OVPRPPOVAXRCED-NQXXGFSBSA-K,OVPRPPOVAXRCED,OVPRPPOVAXRCED-NQXXGFSBSA,2V82,2V82_2,KDP,C([C@H]([C@@H](COP(=O)(O)O)O)O)C(=O)C(=O)O,OVPRPPOVAXRCED-NQXXGFSBSA-N,OVPRPPOVAXRCED
1410,71862,MARCERLRGAALRDVLGRAQGVLFDCDGVLWNGERAVPGAPELLER...,Cc1ncc(COP(=O)([O-])O)c(C=O)c1[O-],1.0,Q96GD0,CHEBI:597326,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,NGVDGCNFYWLIFO-UHFFFAOYSA-L,NGVDGCNFYWLIFO,NGVDGCNFYWLIFO-UHFFFAOYSA,2CFT,2CFT_2,PLP,Cc1c(c(c(cn1)COP(=O)(O)O)C=O)O,NGVDGCNFYWLIFO-UHFFFAOYSA-N,NGVDGCNFYWLIFO
1411,71862,MARCERLRGAALRDVLGRAQGVLFDCDGVLWNGERAVPGAPELLER...,Cc1ncc(COP(=O)([O-])O)c(C=O)c1[O-],1.0,Q96GD0,CHEBI:597326,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,NGVDGCNFYWLIFO-UHFFFAOYSA-L,NGVDGCNFYWLIFO,NGVDGCNFYWLIFO-UHFFFAOYSA,2P69,2P69_3,PLP,Cc1c(c(c(cn1)COP(=O)(O)O)C=O)O,NGVDGCNFYWLIFO-UHFFFAOYSA-N,NGVDGCNFYWLIFO


In [69]:
print(intersection[intersection["Binding"]==1].shape)

(1412, 20)


In [70]:
intersection.to_csv("curated_pdb_intersection_dataset.csv", index=False)

In [88]:
import pandas as pd

intersection = pd.read_csv("curated_pdb_intersection_dataset.csv")

intersection["PDB_ID"] = intersection["PDB_ID"].str.replace('_', '')
intersection

,Unnamed: 0,Sequence,SMILES,Binding,Uniprot_ID,Substrate ID,Publication,Validated,RHEA_ID,EC number,reaction_SMILES,inchikey_x,inchikey_skeleton_x,inchikey_stereochemistry,PDB_ID,nonpol_ID,chem_comp,smiles,inchikey_y,inchikey_skeleton_y
0,6,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C1=CC(=CC=C1C[C@@H](C(=O)O)N)O,1.0,P00509,compound_aminotransferase_dataset 7,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,OUYCCCASQSFEME-QMMMGPOBSA-N,OUYCCCASQSFEME,OUYCCCASQSFEME-QMMMGPOBSA,1AHG,1AHG_2,TYR,c1cc(ccc1C[C@@H](C(=O)O)N)O,OUYCCCASQSFEME-QMMMGPOBSA-N,OUYCCCASQSFEME
1,65914,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,N[C@@H](Cc1ccc(O)cc1)C(=O)O,1.0,P00509,CHEBI:58315,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,OUYCCCASQSFEME-QMMMGPOBSA-N,OUYCCCASQSFEME,OUYCCCASQSFEME-QMMMGPOBSA,1AHG,1AHG_2,TYR,c1cc(ccc1C[C@@H](C(=O)O)N)O,OUYCCCASQSFEME-QMMMGPOBSA-N,OUYCCCASQSFEME
2,18095,MLYIFDLGNVIVDIDFNRVLGAWSDLTRIPLASLKKSFHMGEAFHQ...,O=P(O)(O)O[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@...,1.0,P0A8Y3,compound_phospatase_dataset 156,https://doi.org/10.1073/pnas.1423570112,True,NaN,NaN,NaN,HXXFSFRBOHSIMQ-VFUOTHLCSA-N,HXXFSFRBOHSIMQ,HXXFSFRBOHSIMQ-VFUOTHLCSA,2B0C,2B0C_2,G1P,C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)OP(=O)(O)...,HXXFSFRBOHSIMQ-VFUOTHLCSA-N,HXXFSFRBOHSIMQ
3,70880,MLYIFDLGNVIVDIDFNRVLGAWSDLTRIPLASLKKSFHMGEAFHQ...,O=P([O-])([O-])O[C@H]1O[C@H](CO)[C@@H](O)[C@H]...,1.0,P0A8Y3,CHEBI:58601,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,HXXFSFRBOHSIMQ-VFUOTHLCSA-L,HXXFSFRBOHSIMQ,HXXFSFRBOHSIMQ-VFUOTHLCSA,2B0C,2B0C_2,G1P,C([C@@H]1[C@H]([C@@H]([C@H]([C@H](O1)OP(=O)(O)...,HXXFSFRBOHSIMQ-VFUOTHLCSA-N,HXXFSFRBOHSIMQ
4,28553,MSTPRQILAAIFDMDGLLIDSEPLWDRAELDVMASLGVDISRRNEL...,O=C(O)COP(=O)(O)O,0.0,P77247,compound_phospatase_dataset 54,https://doi.org/10.1073/pnas.1423570112,True,NaN,NaN,NaN,ASCFNMCAHFUBCO-UHFFFAOYSA-N,ASCFNMCAHFUBCO,ASCFNMCAHFUBCO-UHFFFAOYSA,1TE2,1TE2_2,PGA,C(C(=O)O)OP(=O)(O)O,ASCFNMCAHFUBCO-UHFFFAOYSA-N,ASCFNMCAHFUBCO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408,71772,MAAAPRAGRRRGQPLLALLLLLLAPLPPGAPPGADAYFPEERWSPE...,O=c1nc([O-])ccn1[C@@H]1O[C@H](COP(=O)([O-])OP(...,1.0,Q8NBJ5,CHEBI:66914,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,HSCJRCZFDFQWRP-ABVWGUQPSA-L,HSCJRCZFDFQWRP,HSCJRCZFDFQWRP-ABVWGUQPSA,8ZGC,8ZGC_9,GDU,C1=CN(C(=O)NC1=O)[C@H]2[C@@H]([C@@H]([C@H](O2)...,HSCJRCZFDFQWRP-ABVWGUQPSA-N,HSCJRCZFDFQWRP
1409,71800,MQWQTKLPLIAILRGITPDEALAHVGAVIDAGFDAVEIPLNSPQWE...,O=C([O-])C(=O)C[C@@H](O)[C@H](O)COP(=O)([O-])[O-],1.0,Q6BF16,CHEBI:58298,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,OVPRPPOVAXRCED-NQXXGFSBSA-K,OVPRPPOVAXRCED,OVPRPPOVAXRCED-NQXXGFSBSA,2V82,2V82_2,KDP,C([C@H]([C@@H](COP(=O)(O)O)O)O)C(=O)C(=O)O,OVPRPPOVAXRCED-NQXXGFSBSA-N,OVPRPPOVAXRCED
1410,71862,MARCERLRGAALRDVLGRAQGVLFDCDGVLWNGERAVPGAPELLER...,Cc1ncc(COP(=O)([O-])O)c(C=O)c1[O-],1.0,Q96GD0,CHEBI:597326,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,NGVDGCNFYWLIFO-UHFFFAOYSA-L,NGVDGCNFYWLIFO,NGVDGCNFYWLIFO-UHFFFAOYSA,2CFT,2CFT_2,PLP,Cc1c(c(c(cn1)COP(=O)(O)O)C=O)O,NGVDGCNFYWLIFO-UHFFFAOYSA-N,NGVDGCNFYWLIFO
1411,71862,MARCERLRGAALRDVLGRAQGVLFDCDGVLWNGERAVPGAPELLER...,Cc1ncc(COP(=O)([O-])O)c(C=O)c1[O-],1.0,Q96GD0,CHEBI:597326,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,NGVDGCNFYWLIFO-UHFFFAOYSA-L,NGVDGCNFYWLIFO,NGVDGCNFYWLIFO-UHFFFAOYSA,2P69,2P69_3,PLP,Cc1c(c(c(cn1)COP(=O)(O)O)C=O)O,NGVDGCNFYWLIFO-UHFFFAOYSA-N,NGVDGCNFYWLIFO


In [89]:
intersection.drop_duplicates(subset=['Uniprot_ID', "chem_comp"], inplace=True)
intersection.shape

(945, 20)

In [ ]:
import os
import requests
from Bio import PDB
from tqdm import tqdm

def download_pdb(pdb_id, output_dir="."):
    """Download a PDB file from the RCSB."""

    file_path = f"{output_dir}/{pdb_id}.pdb"
    if not os.path.exists(file_path):
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        response = requests.get(url)
        if response.status_code == 200:
            file_path = f"{output_dir}/{pdb_id}.pdb"
            with open(file_path, "wb") as f:
                f.write(response.content)
            return file_path
        else:
            print(f"❌ Failed to download {pdb_id}: {response.status_code}")
            return None
        
    return file_path

def has_ligand(pdb_file, ligand_code):
    """Check if a PDB structure contains a given ligand (3-letter code)."""
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("structure", pdb_file)

    ligand_code = ligand_code.upper()
    for model in structure:
        for chain in model:
            for residue in chain:
                hetfield, resseq, icode = residue.id
                # Ligands are hetero residues (not standard amino acids or water)
                if hetfield.strip() and residue.get_resname() == ligand_code:
                    return True
    return False

os.makedirs("protein_structures", exist_ok=True)

for index, row in tqdm(intersection.iterrows(), total=intersection.shape[0]):
    pdb_id = row['PDB_ID']
    ligand_code = row['chem_comp']  # Assuming 'chem_comp' column has the ligand code
    pdb_file = download_pdb(pdb_id, "protein_structures")
    if pdb_file is not None:
        if not has_ligand(pdb_file, ligand_code):
            print(f"⚠️ {pdb_id} does NOT contain ligand {ligand_code}")


In [91]:
intersection_metadata = intersection[['PDB_ID', 'Uniprot_ID', 'chem_comp', "inchikey_stereochemistry"]]
intersection_metadata.to_csv("curated_pdb_intersection_dataset_metadata.csv", index=False)

In [2]:
import pandas as pd
intersection_metadata = pd.read_csv("curated_pdb_intersection_dataset_metadata.csv")
intersection_metadata

,PDB_ID,Uniprot_ID,chem_comp,inchikey_stereochemistry
0,1AHG,P00509,TYR,OUYCCCASQSFEME-QMMMGPOBSA
1,2B0C,P0A8Y3,G1P,HXXFSFRBOHSIMQ-VFUOTHLCSA
2,1TE2,P77247,PGA,ASCFNMCAHFUBCO-UHFFFAOYSA
3,2E4G,Q8KHZ8,TRP,QIVBCDIJIAJPQS-VIFPVBQESA
4,2PFG,P16152,NAP,XJLXINKUBYWONI-NNYOXOHSSA
...,...,...,...,...
940,2VWH,Q977U7,BGC,WQZGKKKJIJFFOK-VFUOTHLCSA
941,8ZGC,Q8NBJ5,GDU,HSCJRCZFDFQWRP-ABVWGUQPSA
942,2V82,Q6BF16,KDP,OVPRPPOVAXRCED-NQXXGFSBSA
943,2CFT,Q96GD0,PLP,NGVDGCNFYWLIFO-UHFFFAOYSA


In [172]:
import pandas as pd

curated_dataset = pd.read_csv("curated_dataset.csv")
intersection_metadata = pd.read_csv("curated_pdb_intersection_dataset_metadata.csv")
pdb_dataset = curated_dataset[curated_dataset['Enzyme ID'].isin(intersection_metadata["Uniprot_ID"])]
pdb_dataset

,Unnamed: 0,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,Validated,RHEA_ID,EC number,reaction_SMILES
0,0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C(C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
1,1,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C[C@@H](C(=O)O)N,1.0,P00509,compound_aminotransferase_dataset 2,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
2,2,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC(C)[C@@H](C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 3,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
3,3,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC(C)C[C@@H](C(=O)O)N,1.0,P00509,compound_aminotransferase_dataset 4,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
4,4,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC[C@H](C)[C@@H](C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 5,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
69138,71761,MKAIAVKRGEDRPVVIEKPRPEPESGEALVRTLRVGVDGTDHEVIA...,OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O,1.0,Q977U7,C00221,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN
69148,71772,MAAAPRAGRRRGQPLLALLLLLLAPLPPGAPPGADAYFPEERWSPE...,O=c1nc([O-])ccn1[C@@H]1O[C@H](COP(=O)([O-])OP(...,1.0,Q8NBJ5,CHEBI:66914,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN
69168,71800,MQWQTKLPLIAILRGITPDEALAHVGAVIDAGFDAVEIPLNSPQWE...,O=C([O-])C(=O)C[C@@H](O)[C@H](O)COP(=O)([O-])[O-],1.0,Q6BF16,CHEBI:58298,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN
69229,71862,MARCERLRGAALRDVLGRAQGVLFDCDGVLWNGERAVPGAPELLER...,Cc1ncc(COP(=O)([O-])O)c(C=O)c1[O-],1.0,Q96GD0,CHEBI:597326,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN


In [173]:
compounds_intersection_metadata = intersection_metadata[['chem_comp', 'inchikey_stereochemistry']].drop_duplicates()
compounds_intersection_metadata.drop_duplicates(subset=['chem_comp', 'inchikey_stereochemistry'], inplace=True)
compounds_intersection_metadata

,chem_comp,inchikey_stereochemistry
0,TYR,OUYCCCASQSFEME-QMMMGPOBSA
1,G1P,HXXFSFRBOHSIMQ-VFUOTHLCSA
2,PGA,ASCFNMCAHFUBCO-UHFFFAOYSA
3,TRP,QIVBCDIJIAJPQS-VIFPVBQESA
4,NAP,XJLXINKUBYWONI-NNYOXOHSSA
...,...,...
936,ADE,GFFGJBXGBJISGV-UHFFFAOYSA
939,A1EG2,QZDSXQJWBGMRLU-UHFFFAOYSA
942,KDP,OVPRPPOVAXRCED-NQXXGFSBSA
943,PLP,NGVDGCNFYWLIFO-UHFFFAOYSA


In [174]:
pdb_dataset = pdb_dataset.merge(intersection_metadata, left_on=['Enzyme ID'], right_on=['Uniprot_ID'], how='inner')
pdb_dataset

,Unnamed: 0,Sequence,SMILES,Binding,Enzyme ID,Substrate ID,Publication,Validated,RHEA_ID,EC number,reaction_SMILES,PDB_ID,Uniprot_ID,chem_comp,inchikey_stereochemistry
0,0,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C(C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 1,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,1AHG,P00509,TYR,OUYCCCASQSFEME-QMMMGPOBSA
1,1,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,C[C@@H](C(=O)O)N,1.0,P00509,compound_aminotransferase_dataset 2,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,1AHG,P00509,TYR,OUYCCCASQSFEME-QMMMGPOBSA
2,2,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC(C)[C@@H](C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 3,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,1AHG,P00509,TYR,OUYCCCASQSFEME-QMMMGPOBSA
3,3,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC(C)C[C@@H](C(=O)O)N,1.0,P00509,compound_aminotransferase_dataset 4,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,1AHG,P00509,TYR,OUYCCCASQSFEME-QMMMGPOBSA
4,4,MFENITAAPADPILGLADLFRADERPGKINLGIGVYKDETGKTPVL...,CC[C@H](C)[C@@H](C(=O)O)N,0.0,P00509,compound_aminotransferase_dataset 5,https://pubs.acs.org/doi/10.1021/acscatal.0c01895,True,NaN,NaN,NaN,1AHG,P00509,TYR,OUYCCCASQSFEME-QMMMGPOBSA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2210,71761,MKAIAVKRGEDRPVVIEKPRPEPESGEALVRTLRVGVDGTDHEVIA...,OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O,1.0,Q977U7,C00221,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,2VWH,Q977U7,BGC,WQZGKKKJIJFFOK-VFUOTHLCSA
2211,71772,MAAAPRAGRRRGQPLLALLLLLLAPLPPGAPPGADAYFPEERWSPE...,O=c1nc([O-])ccn1[C@@H]1O[C@H](COP(=O)([O-])OP(...,1.0,Q8NBJ5,CHEBI:66914,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,8ZGC,Q8NBJ5,GDU,HSCJRCZFDFQWRP-ABVWGUQPSA
2212,71800,MQWQTKLPLIAILRGITPDEALAHVGAVIDAGFDAVEIPLNSPQWE...,O=C([O-])C(=O)C[C@@H](O)[C@H](O)COP(=O)([O-])[O-],1.0,Q6BF16,CHEBI:58298,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,2V82,Q6BF16,KDP,OVPRPPOVAXRCED-NQXXGFSBSA
2213,71862,MARCERLRGAALRDVLGRAQGVLFDCDGVLWNGERAVPGAPELLER...,Cc1ncc(COP(=O)([O-])O)c(C=O)c1[O-],1.0,Q96GD0,CHEBI:597326,https://doi.org/10.1038/s41467-023-38347-2,True,NaN,NaN,NaN,2CFT,Q96GD0,PLP,NGVDGCNFYWLIFO-UHFFFAOYSA


In [175]:
pdb_dataset = pdb_dataset[["Substrate ID", "SMILES", "Enzyme ID", "PDB_ID", "Binding"]]
pdb_dataset

,Substrate ID,SMILES,Enzyme ID,PDB_ID,Binding
0,compound_aminotransferase_dataset 1,C(C(=O)O)N,P00509,1AHG,0.0
1,compound_aminotransferase_dataset 2,C[C@@H](C(=O)O)N,P00509,1AHG,1.0
2,compound_aminotransferase_dataset 3,CC(C)[C@@H](C(=O)O)N,P00509,1AHG,0.0
3,compound_aminotransferase_dataset 4,CC(C)C[C@@H](C(=O)O)N,P00509,1AHG,1.0
4,compound_aminotransferase_dataset 5,CC[C@H](C)[C@@H](C(=O)O)N,P00509,1AHG,0.0
...,...,...,...,...,...
2210,C00221,OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O,Q977U7,2VWH,1.0
2211,CHEBI:66914,O=c1nc([O-])ccn1[C@@H]1O[C@H](COP(=O)([O-])OP(...,Q8NBJ5,8ZGC,1.0
2212,CHEBI:58298,O=C([O-])C(=O)C[C@@H](O)[C@H](O)COP(=O)([O-])[O-],Q6BF16,2V82,1.0
2213,CHEBI:597326,Cc1ncc(COP(=O)([O-])O)c(C=O)c1[O-],Q96GD0,2CFT,1.0


In [176]:
pdb_dataset["inchikey"] = pdb_dataset['SMILES'].apply(smiles_to_inchikey)
pdb_dataset["inchikey_stereochemistry"] = pdb_dataset['inchikey'].apply(lambda x: '-'.join(x.split('-')[:2]) if pd.notna(x) else None)
pdb_dataset

/tmp/ipykernel_2577228/2740938594.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pdb_dataset["inchikey"] = pdb_dataset['SMILES'].apply(smiles_to_inchikey)
/tmp/ipykernel_2577228/2740938594.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pdb_dataset["inchikey_stereochemistry"] = pdb_dataset['inchikey'].apply(lambda x: '-'.join(x.split('-')[:2]) if pd.notna(x) else None)


,Substrate ID,SMILES,Enzyme ID,PDB_ID,Binding,inchikey,inchikey_stereochemistry
0,compound_aminotransferase_dataset 1,C(C(=O)O)N,P00509,1AHG,0.0,DHMQDGOQFOQNFH-UHFFFAOYSA-N,DHMQDGOQFOQNFH-UHFFFAOYSA
1,compound_aminotransferase_dataset 2,C[C@@H](C(=O)O)N,P00509,1AHG,1.0,QNAYBMKLOCPYGJ-REOHCLBHSA-N,QNAYBMKLOCPYGJ-REOHCLBHSA
2,compound_aminotransferase_dataset 3,CC(C)[C@@H](C(=O)O)N,P00509,1AHG,0.0,KZSNJWFQEVHDMF-BYPYZUCNSA-N,KZSNJWFQEVHDMF-BYPYZUCNSA
3,compound_aminotransferase_dataset 4,CC(C)C[C@@H](C(=O)O)N,P00509,1AHG,1.0,ROHFNLRQFUQHCH-YFKPBYRVSA-N,ROHFNLRQFUQHCH-YFKPBYRVSA
4,compound_aminotransferase_dataset 5,CC[C@H](C)[C@@H](C(=O)O)N,P00509,1AHG,0.0,AGPKZVBTJJNPAG-WHFBIAKZSA-N,AGPKZVBTJJNPAG-WHFBIAKZSA
...,...,...,...,...,...,...,...
2210,C00221,OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O,Q977U7,2VWH,1.0,WQZGKKKJIJFFOK-VFUOTHLCSA-N,WQZGKKKJIJFFOK-VFUOTHLCSA
2211,CHEBI:66914,O=c1nc([O-])ccn1[C@@H]1O[C@H](COP(=O)([O-])OP(...,Q8NBJ5,8ZGC,1.0,HSCJRCZFDFQWRP-ABVWGUQPSA-L,HSCJRCZFDFQWRP-ABVWGUQPSA
2212,CHEBI:58298,O=C([O-])C(=O)C[C@@H](O)[C@H](O)COP(=O)([O-])[O-],Q6BF16,2V82,1.0,OVPRPPOVAXRCED-NQXXGFSBSA-K,OVPRPPOVAXRCED-NQXXGFSBSA
2213,CHEBI:597326,Cc1ncc(COP(=O)([O-])O)c(C=O)c1[O-],Q96GD0,2CFT,1.0,NGVDGCNFYWLIFO-UHFFFAOYSA-L,NGVDGCNFYWLIFO-UHFFFAOYSA


In [177]:
pdb_dataset = pdb_dataset.drop_duplicates(subset=['Substrate ID', 'Enzyme ID', "Binding"])

In [178]:
result = pdb_dataset.merge(compounds_intersection_metadata, left_on=['inchikey_stereochemistry'], right_on=['inchikey_stereochemistry'], how='left')

In [179]:
result = result[['Substrate ID', "SMILES", 'Enzyme ID', 'PDB_ID', 'chem_comp', 'Binding']]

In [180]:

result.columns = ["Ligand_ID", "SMILES", "Uniprot_ID", "PDB_ID", 'PDB_ligand_id', "Binding"]
result.to_csv("curated_pdb_intersection_dataset_all_ligands.csv", index=False)

In [2]:
import pandas as pd


df = pd.read_csv("curated_pdb_intersection_dataset_all_ligands.csv")
df.Binding.value_counts()

1.0    1873
0.0     282
Name: Binding, dtype: int64